In [1]:
# fix for some jupyter event loop problems
import nest_asyncio
nest_asyncio.apply()

In [2]:
import numpy as np
import napari
#from napari.qt.threading import WorkerBase
from napari.qt import thread_worker

import lightsheet
import time

cam_buffer_name = 'HamamatsuFluorBuffer0'
frames_per_volume = 61

In [3]:
def update_volume(frame):
    global viewer
    if frame is None:
        return
    n, data = frame
    z = n % frames_per_volume
    if z == frames_per_volume-1:
        # trigger data changed event
        viewer.layers[0].data = viewer.layers[0].data
        viewer.layers[1].data = viewer.layers[1].data
        viewer.layers[2].data = viewer.layers[2].data
    else:
        # just copy buffer data to image layer
        channels = np.hsplit(data,3)
        viewer.layers[0].data[z,:] = channels[0]
        viewer.layers[1].data[z,:] = channels[1]
        viewer.layers[2].data[z,:] = channels[2]
        

In [4]:
@thread_worker(connect={'yielded': update_volume,
                       'started': (lambda : print('Watcher started...')),
                       'finished': (lambda : print('Watcher stopped.'))})
def watch_buffer(buffer_name):
    with lightsheet.ImageBufferView(buffer_name) as buffer:
        handled = 0
        while True:
            if handled > buffer.written():
                # assume buffer has reset to 0 and may have started writing again
                # could also be we opened a buffer in progress
                handled = max(0,buffer.written()-frames_per_volume)
            
            if handled < buffer.written():
                yield (handled, buffer.get_frame(handled))
                handled = handled + 1
            else:
                time.sleep(.01)
                yield None



In [5]:
viewer = napari.Viewer()

In [6]:
#layer_red = viewer.add_image(np.zeros((30,256,512), dtype='uint16'), name='Red', scale=(2.52,.36,.36), colormap='magma', contrast_limits=[100,3000], blending='additive')
#layer_green = viewer.add_image(np.zeros((30,256,512), dtype='uint16'), name='Green', scale=(2.52,.36,.36), colormap='magma', contrast_limits=[100,2000], blending='additive')

layer_ch1 = viewer.add_image(np.zeros((frames_per_volume-1,256,512), dtype='uint16'), name='698-70', scale=(1.25,.36,.36), colormap='blue', blending='additive')
layer_ch2 = viewer.add_image(np.zeros((frames_per_volume-1,256,512), dtype='uint16'), name='609-54', scale=(1.25,.36,.36), colormap='red', blending='additive')
layer_ch3 = viewer.add_image(np.zeros((frames_per_volume-1,256,512), dtype='uint16'), name='525-50', scale=(1.25,.36,.36), colormap='green', blending='additive')

viewer.grid.enabled = True

In [7]:
worker = watch_buffer(cam_buffer_name)


In [8]:
worker


In [9]:
realdims = np.array([60*1.25,256*.36,512*.36])
line1 = np.array([[.5,.5,0],[.5,.5,1]]) * realdims
line2 = np.array([[.5,0,.5],[.5,1,.5]]) * realdims
line3 = np.array([[0,.5,.5],[1,.5,.5]]) * realdims
shapes_layer = viewer.add_shapes([line1,line2,line3], shape_type='line', name = 'crosshairs', edge_width = 0.36, edge_color='cyan')

Watcher started...
